# Scam text filter
Track: Finance. Prototype only, not financial advice.
Run all cells top to bottom (Runtime > Run all), then save with outputs.

## 1. Load data (SMS Spam Collection, 5,574 messages)

In [ ]:
import urllib.request, zipfile, io
import pandas as pd
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
df = pd.read_csv(z.open("SMSSpamCollection"), sep="\t", names=["label", "text"])
df["y"] = (df.label == "spam").astype(int)
print(df.shape, "share spam:", round(df.y.mean(), 3))
df.head()

## 2. Split + baseline (always predict "not spam")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
X_train, X_test, y_train, y_test = train_test_split(
    df.text, df.y, test_size=0.2, stratify=df.y, random_state=42)
base = [0] * len(y_test)
print("BASELINE accuracy:", round(accuracy_score(y_test, base), 3),
      "| spam recall:", recall_score(y_test, base))

## 3. Model (TF-IDF + logistic regression), scored on the held-out test set

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("MODEL accuracy:", round(accuracy_score(y_test, pred), 3))
print("spam precision:", round(precision_score(y_test, pred), 3))
print("spam recall:   ", round(recall_score(y_test, pred), 3))

## 4. Where it fails (5 wrong messages)

In [ ]:
wrong = X_test[pred != y_test].head(5)
for idx, text in wrong.items():
    print("TRUE:", "spam" if y_test[idx] else "not spam", "|", text, "\n")

## 5. Why does it flag a message? (words that push toward spam)

In [ ]:
import numpy as np
vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
words = np.array(vec.get_feature_names_out())
print("Most spam-like words:", list(words[np.argsort(clf.coef_[0])[-15:]][::-1]))

## 6. OUR OWN ADDITION: modern scams
TODO: replace these examples with 20+ messages YOUR TEAM wrote or collected (scams common where you live today, plus normal texts). Never use real people's messages without permission. Label 1 = scam, 0 = normal.

In [ ]:
ours = [
    ("Your package is on hold. Pay $1.99 at http://redeliver-now.example to release it", 1),
    ("Hey are you coming to practice at 4?", 0),
]
texts, labels = zip(*ours)
our_pred = model.predict(list(texts))
print("accuracy on our messages:", accuracy_score(labels, our_pred))
print("spam recall on our messages:", recall_score(labels, our_pred))
for t, l, p in zip(texts, labels, our_pred):
    print("OK " if l == p else "MISS", "|", t)